# 02 Vintage Curves

**Purpose:** Cumulative default and prepayment % by vintage quarter and MOB.

**Steps**
- Cumulative default % by vintage x MOB
- Cumulative prepay %
- Marginal default rate
- Curves, heatmap triangle
- Fixed-MOB comparison (12/24/36/60)

**Read first:** Bellini 3.2, 3.4, then 5.4 (competing risks)

**Done when:** fixed-MOB table for the 2007 vintages plus a written read of the curves.

**Year in this pass:** 2007 only. Functions that work get moved into `src/freddie/`.

## 0. Setup and Data Loading

In [1]:
import os
import sys

import pandas as pd
import numpy as np
import matplotlib
import seaborn
import plotly


sys.path.append(r"C:\Dev\CreditRiskLearning\Projects\Freddie Mac End-to-End Project\src")
from freddie.config import RAW_DIR, INTERIM_DIR, PROCESSED_DIR



In [2]:
# Vintage years

years = [2006, 2007, 2010, 2016]

loan_table = {}

In [3]:
# Load interim parquets

for year in years:
    
    loan_table[year] = pd.read_parquet(PROCESSED_DIR /f"loan_table_{year}.parquet" )
    

In [4]:
# Make whole vintage table

vintage_table = pd.concat([loan_table[2006], loan_table[2007], loan_table[2010], loan_table[2016]], ignore_index=True)


## 1. Cumulative Default Rate

**Concept:** Count, Fill, Accumulate and Divide

By applying Equation 1.1 we can determine the cumulative default rate per vintage cohort. Equation 1.1 in laymans terms states to go through every loan in the vintage. Score 1 if it defaulted by month m, otherwise 0. Divide the total score by the number of loans.

$$F(m) = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1}\left[\text{outcome}_i = \text{default} \;\wedge\; \text{event\_mob}_i \le m\right] \tag{1.1}$$




| Symbol | Meaning |
| :--- | :--- |
| $F(m)$ | Cumulative default rate for the vintage at month-on-book $m$ |
| $m$ | A month on book (MOB), e.g. 6 or 12 |
| $i$ | One loan in the vintage |
| $N$ | Number of loans in the vintage, counting every loan: defaulted, prepaid, open and censored |
| $\text{outcome}_i$ | The loan's outcome: default, prepay, open or censored |
| $\text{event\_mob}_i$ | The MOB at which the loan's first event happened |
| $1[\cdot]$ | Indicator: 1 if the condition inside is true, 0 otherwise |
| $\wedge$ | "and": both conditions must hold |


**Note:** $\text{event\_mob}_i$ is udefined for open loans. Prepaid loans have an event MOB, but it doesn’t count as a default.

**Assumption:** For $N$, assume each loan is observed to at least m months. The truncation rule for vintages that haven’t reached m is set out below.

In [5]:
vintage_table.head(5)

,vintage,last_mob,default_mob,prepay_mob,censor_mob,first_event,event_mob
0,2006Q1,187,NaN,187.0,NaN,prepaid,187.0
1,2006Q1,34,NaN,34.0,NaN,prepaid,34.0
2,2006Q1,36,NaN,36.0,NaN,prepaid,36.0
3,2006Q1,40,NaN,40.0,NaN,prepaid,40.0
4,2006Q1,8,NaN,8.0,NaN,prepaid,8.0


In [ ]:
# Build the numerator: count loans with first_event == "default" by vintage and event_mob.

vintage_events = pd.DataFrame()

defaults = vintage_table[vintage_table['first_event'] == 'default']

defaults_by_vintage = defaults.groupby(['vintage', 'event_mob']).size()

In [14]:
# Build the denominator: count all loans per vintage in loan_table, including prepaid and open loans.

loan_count_vintage = vintage_table.groupby('vintage').size()

In [ ]:
# Reindex to every MOB in the range with fill 0, then cumsum along MOB.

defaults_by_vintage.reindex('event_mob')

vintage  event_mob
2006Q1   0.0          1
         3.0          6
         4.0          6
         5.0          3
         6.0          3
dtype: int64

In [ ]:
# Apply your truncation rule so each vintage only shows MOBs it has been observed for.

In [ ]:
# Print the 2006 curve and send it to me.